<a href="https://colab.research.google.com/github/sumin031130/ESAA/blob/main/ESAA_%E1%84%89%E1%85%AE%E1%84%89%E1%85%A1%E1%86%BC%E1%84%8C%E1%85%A1%E1%86%A8%E1%84%85%E1%85%B5%E1%84%87%E1%85%B2_KBO%E1%84%8C%E1%85%A6%E1%84%80%E1%85%AE%E1%84%89%E1%85%A5%E1%86%BC%E1%84%80%E1%85%A9%E1%86%BC_RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [ESAA 수상작 리뷰] KBO 제구 성공 확률 예측 – Baseline RandomForest

## 0. 리뷰 대상

| 항목 | 내용 |
|---|---|
| 대회 | KBO 투구의 제구 성공 확률 예측 |
| 대회 링크 | https://dacon.io/competitions/official/236743/codeshare/14147?page=1&dtype=recent$0 |
| 입력 | test.csv의 47개 컬럼(경기 상황, 투수·타자의 직전까지 누적 기록 등)|
| 출력 | 제구 성공 확률 (0 이상 1 이하의 실수) |
| 평가지표 | Brier Skill Score |

## 1. 데이터

- **문제**: KBO 투구 하나가 **제구 성공** 투구일 확률을 예측한다. 타깃은 `control_success`(0/1)이고 출력은 0 이상 1 이하의 확률이다.
- **파일**
  - `train.csv`: 2019~2024 시즌. 평가 데이터(`test.csv`)는 2025 시즌이다.
  - `test.csv`: 입력 컬럼이 들어 있고, 모델이 쓸 피처 목록을 정한다.
  - `trackman_history.csv`: 2019~2024 과거 로그 약 179만 행. **이 베이스라인에서는 사용하지 않는다.**
- **피처**: 47개(범주형 3개 + 수치형 44개). 경기 상황, 투수·타자의 직전까지 누적 기록 등이다.
  - 범주형: `top_bottom`, `game_type`, `base_state`
  - ID 컬럼: `row_id` (학습에서 제외)
- **평가지표 Brier Skill Score**: 예측 확률과 실제값(0/1) 차이의 제곱 평균(Brier)을 상수 예측의 Brier인 `r(1-r)`로 나눠 계산한다. 값이 작을수록 좋은 Brier를 "기준선 대비 얼마나 좋아졌는지"로 바꾼 점수다.
- 노트북에는 실행 출력이 저장돼 있지 않아서 행 수, 성공률, 검증 점수는 이 리뷰에 적지 않았다.

## 2. 코드 흐름

1. 라이브러리 불러오기 (pandas, scikit-learn, joblib)
2. 데이터 불러오기: `test.csv` 헤더로 피처 목록을 정하고, `train.csv`에서 그 컬럼과 타깃만 읽는다.
3. 전처리 정의: 범주형은 `OrdinalEncoder`, 수치형은 중앙값 대체(`SimpleImputer`)를 `ColumnTransformer`로 묶는다.
4. 모델 정의와 학습: 전처리 + `RandomForestClassifier`를 하나의 `Pipeline`으로 만들고, 2024 시즌을 검증용으로 떼어 2019~2023으로 학습한다.
5. 검증: 2024 시즌에서 Brier와 Validation Score를 계산한다.
6. 전체 데이터로 재학습한 뒤 `./model/rf.pkl`로 저장한다. 추론용 `script.py`, `requirements.txt`와 함께 `baseline_submit.zip`으로 묶어 제출한다.

## 3. 주요 코드 분석

### 3-1. 피처 목록을 test.csv가 정한다

```python
test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),
                        encoding="utf-8-sig", nrows=0).columns
FEATURES = [c for c in test_cols if c != ID]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]
```

- `nrows=0`으로 **헤더만 읽어** 컬럼 이름을 얻는다. 데이터를 읽지 않아 빠르다.
- `train.csv`에만 있는 컬럼을 학습에 넣으면 평가 시점에 그 컬럼이 없어 **추론이 실패**한다.
- 범주형 목록(`CAT_COLS`)만 직접 적고, 나머지는 전부 수치형으로 취급한다.

### 3-2. 필요한 컬럼만 읽기

```python
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"),
                    encoding="utf-8-sig", usecols=FEATURES + [TARGET])
```

- `usecols`로 피처와 타깃만 읽어 메모리를 아낀다. `season`이 피처 목록에 들어 있으므로 검증 분할에도 같은 컬럼을 쓴다.

### 3-3. 전처리를 Pipeline 안에 넣기

```python
preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                           unknown_value=-1), CAT_COLS),
    ("num", SimpleImputer(strategy="median"), NUM_COLS),
])
```

- 범주형은 정수로, 수치형 결측은 중앙값으로 채운다.
- `handle_unknown="use_encoded_value"`는 학습 때 못 본 범주가 평가 데이터에 나오면 **-1**로 처리한다는 뜻이다. 추론 때 에러가 나지 않게 하는 안전장치다.
- 전처리를 모델과 한 `Pipeline`에 묶었기 때문에 **추론할 때 같은 변환이 자동으로 적용**된다. 전처리를 빠뜨리는 실수를 막고, 대체값(중앙값)도 학습 데이터로만 계산된다.

### 3-4. 얕은 랜덤 포레스트

```python
RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=200,
    n_jobs=-1,
    random_state=42,
)
```

- 트리 깊이를 10, 잎 노드 최소 샘플 수를 200으로 제한했다. 노트북 설명에 따르면 학습이 1분 안에 끝나고 모델 파일이 약 4MB로 가볍다.
- `min_samples_leaf=200`은 **잎마다 평균을 200개 이상으로 만들어** 확률 추정을 안정시키는 효과가 있다.
- `random_state=42`로 고정하고 GPU를 쓰지 않으므로, 같은 데이터와 패키지 버전이면 결과가 같다.

### 3-5. 시간 순서로 검증 분할

```python
is_val = train["season"] == 2024
X_train, y_train = train.loc[~is_val, FEATURES], train.loc[~is_val, TARGET]
X_val, y_val = train.loc[is_val, FEATURES], train.loc[is_val, TARGET]
```

- 2024 시즌을 검증용으로 떼고 2019~2023으로 학습한다. 평가 데이터가 **미래 시즌(2025)**이므로 무작위 분할보다 실제 상황에 가깝다.

### 3-6. Brier Skill Score 계산

```python
val_pred = model.predict_proba(X_val)[:, 1]

r = y_val.mean()
brier = ((val_pred - y_val) ** 2).mean()
baseline_brier = r * (1 - r)
score = max(0, 100000 * (1 - brier / baseline_brier))
```

- `predict_proba(...)[:, 1]`이 "제구 성공(1)"일 확률이다.
- 상수 예측(모든 투구에 성공률 `r`을 예측)의 Brier는 `r(1-r)`이다. 이보다 나아진 비율이 `1 - brier / baseline_brier`이고, 여기에 100000을 곱해 점수로 쓴다. 상수 예측보다 나쁘면 0점이다(`max(0, ...)`).
- 즉 **성공률만 찍어도 0점에 가깝고, 피처를 잘 쓸수록 점수가 오른다.**

### 3-7. 전체 데이터로 재학습 후 저장

```python
model.fit(train[FEATURES], train[TARGET])
os.makedirs("./model", exist_ok=True)
joblib.dump(model, "./model/rf.pkl", compress=3)
```

- 검증으로 성능을 확인한 뒤 2024 시즌까지 포함해 다시 학습한다. 최신 시즌 정보를 쓰기 위해서다.
- `joblib.dump(..., compress=3)`으로 파이프라인 전체를 압축 저장한다. 전처리와 모델이 함께 들어 있어 `script.py`가 이 파일 하나만 불러오면 된다.

## 4. 새롭게 알게 된 점
- **전처리를 Pipeline에 넣으면** 추론 때 같은 변환이 따라가고, 교차검증이나 검증 분할에서 누수를 막기도 쉽다.
- **시간 순서 검증**(마지막 시즌을 검증용으로)이 미래를 예측하는 문제에 맞는 방법이다.
- Brier 예측한 확률이 실제 결과와 얼마나 가까웠는지 나타내는 지표.
(이 대회에서는 0에 가까울수록 좋음)
  - Brier 식: (1/n) Σ (p̂ - y)²Brier 식: (1/n) Σ (p̂ - y)²
  - 값이 작을으록 좋음. 정확도와는 다르다.

## 5. 아쉬운 점 / 주의할 점

- **베이스라인이라 개선 여지가 크다.** 피처 엔지니어링, 하이퍼파라미터 튜닝, 다른 모델(LightGBM, XGBoost 등)과의 비교가 없다. 설정(`n_estimators=100`, `max_depth=10`, `min_samples_leaf=200`)의 근거도 적혀 있지 않다.
- **검증 점수만 보고 끝난다.** 노트북에 저장된 출력이 없어서 이 리뷰에서 점수를 확인하지 못했다. 시즌 한 번(2024)으로만 검증해서 점수가 그 시즌에 치우칠 수 있다.
- **확률 보정(calibration)을 하지 않는다.** 랜덤 포레스트는 확률이 극단값에 덜 가는 경향이 있어 Brier를 줄이려면 보정이나 다른 모델을 비교할 여지가 있다